# PySpark practice 3: types, casting and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-03-types-casting-and-nulls.ipynb)

14 exercises on types, casting under ANSI mode, decimals, time zones, null semantics, NaN and reject handling. Each exercise names the [reference entries](pyspark-reference-03-types-casting-and-nulls.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 3](pyspark-reference-03-types-casting-and-nulls.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

6 easy, 7 medium, 1 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Type raw strings and flag rejects](#exercise-1-type-raw-strings-and-flag-rejects) | Medium | [`Column.try_cast(dataType)`](pyspark-reference-03-types-casting-and-nulls.ipynb#columntry_castdatatype) |
| 2 | [Parse formatted amounts](#exercise-2-parse-formatted-amounts) | Easy | [`to_number`, `try_to_number` and format strings](pyspark-reference-03-types-casting-and-nulls.ipynb#to_number-try_to_number-and-format-strings) |
| 3 | [Money in decimal, not double](#exercise-3-money-in-decimal-not-double) | Easy | [Decimal precision, scale and arithmetic](pyspark-reference-03-types-casting-and-nulls.ipynb#decimal-precision-scale-and-arithmetic) |
| 4 | [Survive overflow and division by zero](#exercise-4-survive-overflow-and-division-by-zero) | Medium | [ANSI mode](pyspark-reference-03-types-casting-and-nulls.ipynb#ansi-mode) |
| 5 | [Detect changed attributes null-safely](#exercise-5-detect-changed-attributes-null-safely) | Medium | [`isNull`, `isNotNull` and `eqNullSafe`](pyspark-reference-03-types-casting-and-nulls.ipynb#isnull-isnotnull-and-eqnullsafe) |
| 6 | [Rows lost to three-valued logic](#exercise-6-rows-lost-to-three-valued-logic) | Easy | [Three-valued logic](pyspark-reference-03-types-casting-and-nulls.ipynb#three-valued-logic) |
| 7 | [Fill nulls with type-appropriate defaults](#exercise-7-fill-nulls-with-type-appropriate-defaults) | Medium | [`fillna(value, subset=None)` / `df.na.fill`](pyspark-reference-03-types-casting-and-nulls.ipynb#fillnavalue-subsetnone--dfnafill) |
| 8 | [Drop rows missing all contact details](#exercise-8-drop-rows-missing-all-contact-details) | Easy | [`dropna(how="any", thresh=None, subset=None)` / `df.na.drop`](pyspark-reference-03-types-casting-and-nulls.ipynb#dropnahowany-threshnone-subsetnone--dfnadrop) |
| 9 | [Turn source sentinels into nulls](#exercise-9-turn-source-sentinels-into-nulls) | Easy | [`replace(to_replace, value, subset=None)` / `df.na.replace`](pyspark-reference-03-types-casting-and-nulls.ipynb#replaceto_replace-value-subsetnone--dfnareplace) |
| 10 | [Average a sensor feed with NaN and null](#exercise-10-average-a-sensor-feed-with-nan-and-null) | Medium | [NaN semantics](pyspark-reference-03-types-casting-and-nulls.ipynb#nan-semantics) |
| 11 | [Count by a nullable key](#exercise-11-count-by-a-nullable-key) | Easy | [How nulls behave across operations](pyspark-reference-03-types-casting-and-nulls.ipynb#how-nulls-behave-across-operations) |
| 12 | [Same instant, different wall clocks](#exercise-12-same-instant-different-wall-clocks) | Medium | [`timestamp` vs `timestamp_ntz`](pyspark-reference-03-types-casting-and-nulls.ipynb#timestamp-vs-timestamp_ntz) |
| 13 | [Report runtime types of expressions](#exercise-13-report-runtime-types-of-expressions) | Medium | [Decimal precision, scale and arithmetic](pyspark-reference-03-types-casting-and-nulls.ipynb#decimal-precision-scale-and-arithmetic) |
| 14 | [Quarantine rows that fail typing](#exercise-14-quarantine-rows-that-fail-typing) | Hard | [Typing raw data with a reject count](pyspark-reference-03-types-casting-and-nulls.ipynb#typing-raw-data-with-a-reject-count) |

## Exercise 1: Type raw strings and flag rejects

**Difficulty:** Medium · about 10 min · **Practises:** [`Column.try_cast(dataType)`](pyspark-reference-03-types-casting-and-nulls.ipynb#columntry_castdatatype)

*Bronze-to-silver typing must separate missing values from invalid ones.*

Cast `qty` to `int` and `price` to `decimal(10,2)`. Return `id`, `qty`, `price` and `rejected` (true when any present raw value failed to convert), ordered by `id`.

In [ ]:
raw = spark.createDataFrame([("a", "3", "9.99"), ("b", "three", "5.00"), ("c", None, "1,200.00"), ("d", "2", None)], "id string, qty string, price string")
raw.show()

**Expected output**

| id | qty | price | rejected |
| --- | --- | --- | --- |
| a | 3 | 9.99 | false |
| b | null | 5.00 | true |
| c | null | null | true |
| d | 2 | null | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
typed = raw.select("id", F.col("qty").try_cast("int").alias("qty"), F.col("price").try_cast("decimal(10,2)").alias("price"),
                   F.col("qty").alias("raw_qty"), F.col("price").alias("raw_price"))
rejected = (F.col("raw_qty").isNotNull() & F.col("qty").isNull()) | (F.col("raw_price").isNotNull() & F.col("price").isNull())
result = typed.select("id", "qty", "price", rejected.alias("rejected")).orderBy("id")
result.show()

## Exercise 2: Parse formatted amounts

**Difficulty:** Easy · about 5 min · **Practises:** [`to_number`, `try_to_number` and format strings](pyspark-reference-03-types-casting-and-nulls.ipynb#to_number-try_to_number-and-format-strings)

*Finance exports format numbers for humans; pipelines must parse them back.*

Parse `text` with format `9,999.99` into `amount` (null when it does not match). Return `text` and `amount`, in input order.

In [ ]:
amounts = spark.createDataFrame([(1, "1,204.50"), (2, "99.00"), (3, "7"), (4, "N/A")], "n int, text string")
amounts.show()

**Expected output**

| text | amount |
| --- | --- |
| 1,204.50 | 1204.50 |
| 99.00 | 99.00 |
| 7 | 7.00 |
| N/A | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = amounts.orderBy("n").select("text", F.try_to_number("text", F.lit("9,999.99")).alias("amount"))
result.show()

## Exercise 3: Money in decimal, not double

**Difficulty:** Easy · about 5 min · **Practises:** [Decimal precision, scale and arithmetic](pyspark-reference-03-types-casting-and-nulls.ipynb#decimal-precision-scale-and-arithmetic)

*Revenue reconciliations fail by a cent when money is summed as double.*

Sum the payments of 0.10 and 0.20 as `double` and as `decimal(10,2)`. Return `type`, `total` (as a string) and `equals_expected` (whether the total equals 0.30).

In [ ]:
payments = spark.createDataFrame([(0.1, D("0.10")), (0.2, D("0.20"))], "as_double double, as_decimal decimal(10,2)")
payments.show()

**Expected output**

| type | total | equals_expected |
| --- | --- | --- |
| double | 0.30000000000000004 | false |
| decimal | 0.30 | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
totals = payments.agg(F.sum("as_double").alias("d"), F.sum("as_decimal").alias("m")).first()
result = spark.createDataFrame([
    ("double", str(totals.d), totals.d == 0.3),
    ("decimal", str(totals.m), totals.m == D("0.30")),
], "type string, total string, equals_expected boolean")
result.show()

## Exercise 4: Survive overflow and division by zero

**Difficulty:** Medium · about 10 min · **Practises:** [ANSI mode](pyspark-reference-03-types-casting-and-nulls.ipynb#ansi-mode)

*Spark 4 fails the job on overflow; the try_ functions keep it running and visible.*

For each row compute `ratio` = `numerator / denominator` and `doubled` = `numerator * 2` without failing: use null where the operation is invalid. Return `id`, `ratio`, `doubled`, ordered by `id`.

In [ ]:
ops = spark.createDataFrame([(1, 10, 4), (2, 5, 0), (3, 2147483647, 1)], "id int, numerator int, denominator int")
ops.show()

**Expected output**

| id | ratio | doubled |
| --- | --- | --- |
| 1 | 2.5 | 20 |
| 2 | null | 10 |
| 3 | 2147483647.0 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = ops.select("id", F.try_divide("numerator", "denominator").alias("ratio"), F.try_multiply("numerator", F.lit(2)).alias("doubled")).orderBy("id")
result.show()

## Exercise 5: Detect changed attributes null-safely

**Difficulty:** Medium · about 10 min · **Practises:** [`isNull`, `isNotNull` and `eqNullSafe`](pyspark-reference-03-types-casting-and-nulls.ipynb#isnull-isnotnull-and-eqnullsafe)

*Change detection with `!=` misses changes to and from null.*

Return `id` and `changed` for every row of `snap`, where `changed` is true when `old` and `new` differ, treating two nulls as equal and null vs a value as different. Order by `id`.

In [ ]:
snap = spark.createDataFrame([(1, "gold", "gold"), (2, "gold", "silver"), (3, None, "gold"), (4, "gold", None), (5, None, None)], "id int, old string, new string")
snap.show()

**Expected output**

| id | changed |
| --- | --- |
| 1 | false |
| 2 | true |
| 3 | true |
| 4 | true |
| 5 | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = snap.select("id", (~F.col("old").eqNullSafe(F.col("new"))).alias("changed")).orderBy("id")
result.show()

## Exercise 6: Rows lost to three-valued logic

**Difficulty:** Easy · about 5 min · **Practises:** [Three-valued logic](pyspark-reference-03-types-casting-and-nulls.ipynb#three-valued-logic)

*A filter and its negation should cover every row; with nulls they do not.*

Return one row with `total`, `matches` (region = 'West'), `non_matches` (NOT region = 'West') and `lost` (rows in neither) for `customers`.

In [ ]:
customers.select("customer_id", "region").show()

**Expected output**

| total | matches | non_matches | lost |
| --- | --- | --- | --- |
| 8 | 2 | 5 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
total = customers.count()
matches = customers.where(F.col("region") == "West").count()
non_matches = customers.where(~(F.col("region") == "West")).count()
result = spark.createDataFrame([(total, matches, non_matches, total - matches - non_matches)], "total int, matches int, non_matches int, lost int")
result.show()

## Exercise 7: Fill nulls with type-appropriate defaults

**Difficulty:** Medium · about 10 min · **Practises:** [`fillna(value, subset=None)` / `df.na.fill`](pyspark-reference-03-types-casting-and-nulls.ipynb#fillnavalue-subsetnone--dfnafill)

*A scalar fillna silently skips columns of other types.*

Fill `channel` with `unknown`, `qty` with 0 and leave `score` null. Return all columns ordered by `id`.

In [ ]:
gaps = spark.createDataFrame([(1, None, None, None), (2, "web", 5, 1.5), (3, None, 2, None)], "id int, channel string, qty int, score double")
gaps.show()

**Expected output**

| id | channel | qty | score |
| --- | --- | --- | --- |
| 1 | unknown | 0 | null |
| 2 | web | 5 | 1.5 |
| 3 | unknown | 2 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = gaps.fillna({"channel": "unknown", "qty": 0}).orderBy("id")
result.show()

## Exercise 8: Drop rows missing all contact details

**Difficulty:** Easy · about 5 min · **Practises:** [`dropna(how="any", thresh=None, subset=None)` / `df.na.drop`](pyspark-reference-03-types-casting-and-nulls.ipynb#dropnahowany-threshnone-subsetnone--dfnadrop)

*Contact lists keep a person if any channel to reach them exists.*

Remove rows where both `phone` and `email` are null. Return `id`, ordered.

In [ ]:
contacts = spark.createDataFrame([(1, "555-01", None), (2, None, None), (3, None, "a@x.com"), (4, None, None)], "id int, phone string, email string")
contacts.show()

**Expected output**

| id |
| --- |
| 1 |
| 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = contacts.dropna(how="all", subset=["phone", "email"]).select("id").orderBy("id")
result.show()

## Exercise 9: Turn source sentinels into nulls

**Difficulty:** Easy · about 5 min · **Practises:** [`replace(to_replace, value, subset=None)` / `df.na.replace`](pyspark-reference-03-types-casting-and-nulls.ipynb#replaceto_replace-value-subsetnone--dfnareplace)

*Legacy systems write -1, empty strings and 'N/A' for missing values.*

Replace `N/A`, `n/a` and empty strings in `channel` with null, and `-1` in `qty` with null. Return all columns ordered by `id`.

In [ ]:
sentinels = spark.createDataFrame([(1, "N/A", -1), (2, "web", 3), (3, "", -1), (4, "n/a", 0)], "id int, channel string, qty int")
sentinels.show()

**Expected output**

| id | channel | qty |
| --- | --- | --- |
| 1 | null | null |
| 2 | web | 3 |
| 3 | null | null |
| 4 | null | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = sentinels.replace(["N/A", "n/a", ""], None, subset=["channel"]).replace(-1, None, subset=["qty"]).orderBy("id")
result.show()

## Exercise 10: Average a sensor feed with NaN and null

**Difficulty:** Medium · about 10 min · **Practises:** [NaN semantics](pyspark-reference-03-types-casting-and-nulls.ipynb#nan-semantics)

*One NaN poisons every aggregate over it.*

Return one row with `rows`, `missing` (null), `invalid` (NaN), `usable` and `avg_usable` (average of values that are neither null nor NaN) for `sensor`.

In [ ]:
sensor = spark.createDataFrame([(12.5,), (None,), (float("nan"),), (0.0,), (7.5,)], "reading double")
sensor.show()

**Expected output**

| rows | missing | invalid | usable | avg_usable |
| --- | --- | --- | --- | --- |
| 5 | 1 | 1 | 3 | 6.666666666666667 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
usable = F.nanvl("reading", F.lit(None).cast("double"))
result = sensor.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("reading").isNull()).alias("missing"),
    F.count_if(F.isnan("reading")).alias("invalid"),
    F.count(usable).alias("usable"),
    F.avg(usable).alias("avg_usable"),
)
result.show()

## Exercise 11: Count by a nullable key

**Difficulty:** Easy · about 5 min · **Practises:** [How nulls behave across operations](pyspark-reference-03-types-casting-and-nulls.ipynb#how-nulls-behave-across-operations)

*Null group keys form their own group; reports should label it.*

Count customers per region, labelling the null region `unknown`. Return `region` and `customers`, ordered by `region`.

In [ ]:
customers.select("customer_id", "region").show()

**Expected output**

| region | customers |
| --- | --- |
| East | 2 |
| North | 2 |
| South | 1 |
| West | 2 |
| unknown | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = customers.groupBy(F.coalesce("region", F.lit("unknown")).alias("region")).agg(F.count("*").alias("customers")).orderBy("region")
result.show()

## Exercise 12: Same instant, different wall clocks

**Difficulty:** Medium · about 10 min · **Practises:** [`timestamp` vs `timestamp_ntz`](pyspark-reference-03-types-casting-and-nulls.ipynb#timestamp-vs-timestamp_ntz)

*Session time zones silently shift every derived date of a timestamp column.*

For each session time zone in `zones`, return `zone`, `ts_text` (the `timestamp` column cast to string) and `ntz_text` (the `timestamp_ntz` column cast to string). Restore UTC afterwards.

In [ ]:
zones = ["UTC", "Asia/Kolkata", "America/New_York"]
both = spark.createDataFrame([(datetime(2026, 1, 14, 0, 30), datetime(2026, 1, 14, 0, 30))], "ts timestamp, ntz timestamp_ntz")

**Expected output**

| zone | ts_text | ntz_text |
| --- | --- | --- |
| UTC | 2026-01-14 00:30:00 | 2026-01-14 00:30:00 |
| Asia/Kolkata | 2026-01-14 06:00:00 | 2026-01-14 00:30:00 |
| America/New_York | 2026-01-13 19:30:00 | 2026-01-14 00:30:00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rows = []
for zone in zones:
    spark.conf.set("spark.sql.session.timeZone", zone)
    r = both.select(F.col("ts").cast("string").alias("ts_text"), F.col("ntz").cast("string").alias("ntz_text")).first()
    rows.append((zone, r.ts_text, r.ntz_text))
spark.conf.set("spark.sql.session.timeZone", "UTC")
result = spark.createDataFrame(rows, "zone string, ts_text string, ntz_text string")
result.show()

## Exercise 13: Report runtime types of expressions

**Difficulty:** Medium · about 10 min · **Practises:** [Decimal precision, scale and arithmetic](pyspark-reference-03-types-casting-and-nulls.ipynb#decimal-precision-scale-and-arithmetic)

*Implicit widening and double literals change result types; typeof shows what you really get.*

For the expressions in `exprs`, return `expression` and `type` (from SQL `typeof`), in the given order.

In [ ]:
exprs = ["amount + amount", "amount * amount", "amount * 1.2", "amount * 1.2D", "sum(amount)", "avg(amount)"]
print(exprs)

**Expected output**

| expression | type |
| --- | --- |
| amount + amount | decimal(13,2) |
| amount * amount | decimal(25,4) |
| amount * 1.2 | decimal(15,3) |
| amount * 1.2D | double |
| sum(amount) | decimal(22,2) |
| avg(amount) | decimal(16,6) |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rows = []
for e in exprs:
    agg = e.startswith(("sum", "avg"))
    t = (orders.agg(F.expr(f"typeof({e})")) if agg else orders.select(F.expr(f"typeof({e})"))).first()[0]
    rows.append((e, t))
result = spark.createDataFrame(rows, "expression string, type string")
result.show(truncate=False)

## Exercise 14: Quarantine rows that fail typing

**Difficulty:** Hard · about 15 min · **Practises:** [Typing raw data with a reject count](pyspark-reference-03-types-casting-and-nulls.ipynb#typing-raw-data-with-a-reject-count)

*Production pipelines route bad rows to a quarantine table and keep loading the good ones.*

Type `landing` (`qty` int, `order_date` date, `amount` decimal(10,2)). Return `order_id`, `bad_columns` (comma-separated names of present values that failed to convert, in column order) for rejected rows only, ordered by `order_id`.

In [ ]:
landing = spark.createDataFrame([("o1", "4", "2026-01-05", "10.00"), ("o2", "x", "2026-01-06", "abc"), ("o3", "2", "05/01/2026", "3.50"), ("o4", None, None, None)],
                                "order_id string, qty string, order_date string, amount string")
landing.show()

**Expected output**

| order_id | bad_columns |
| --- | --- |
| o2 | qty,amount |
| o3 | order_date |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
targets = {"qty": "int", "order_date": "date", "amount": "decimal(10,2)"}
failures = [F.when(F.col(c).isNotNull() & F.col(c).try_cast(t).isNull(), F.lit(c)) for c, t in targets.items()]
result = (
    landing.select("order_id", F.concat_ws(",", *failures).alias("bad_columns"))
    .where(F.col("bad_columns") != "")
    .orderBy("order_id")
)
result.show()

**Next:** look anything up in [PySpark reference 3: types, casting and nulls](pyspark-reference-03-types-casting-and-nulls.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).